# Medical Continued Pretraining (CPT) Pipeline
## Qwen2.5-7B on Augmented Medical Data

**Hardware Requirements:**
- GPU with ≥24GB VRAM (RTX 4090, A100, RTX 5090, etc.)
- Ubuntu 20.04+ with CUDA 12.1+
- 100GB+ free disk space
- Python 3.10+

**Expected Training Time:**
- ~8-12 hours on RTX 4090/5090
- Training will show progress every 10 steps

## 1. Environment Setup

In [ ]:
import os
import sys
import json
import torch
import numpy as np
from pathlib import Path
from datetime import datetime
from typing import Dict, List

# Print system info
print("\n" + "="*80)
print("SYSTEM INFORMATION")
print("="*80)
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"GPU Compute Capability: {torch.cuda.get_device_capability(0)}")
else:
    print("⚠️  WARNING: CUDA not available! Training will be very slow.")

print("="*80 + "\n")

In [ ]:
# Check required packages
required_packages = {
    'transformers': '4.36+',
    'datasets': '2.14+',
    'torch': '2.0+',
    'peft': 'optional (for LoRA)',
}

print("Checking packages...")
try:
    from transformers import (
        AutoModelForCausalLM,
        AutoTokenizer,
        Trainer,
        TrainingArguments,
        DataCollatorForLanguageModeling,
    )
    from datasets import Dataset
    print("✓ All required packages available")
except ImportError as e:
    print(f"✗ Missing package: {e}")
    print("\nInstall with:")
    print("pip install -q transformers datasets torch")
    sys.exit(1)

print("✓ Ready to train!\n")

## 2. Configuration

In [ ]:
# ============ TRAINING CONFIGURATION ============
CONFIG = {
    # Model & Data
    "model_name": "Qwen/Qwen2.5-7B",
    "train_file": "augmented_output/train.jsonl",
    "eval_file": "augmented_output/eval.jsonl",

    # Training Parameters (balanced for 24GB+ GPU)
    "num_train_epochs": 3,
    "per_device_train_batch_size": 2,  # Per GPU batch size
    "per_device_eval_batch_size": 4,   # Eval batch can be larger
    "gradient_accumulation_steps": 8,  # Accumulate 8 steps → effective batch = 2*8=16
    "learning_rate": 2e-5,
    "warmup_steps": 500,
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,

    # Sequence & Optimization
    "max_seq_length": 2048,
    "bf16": True,  # Mixed precision (bfloat16 for modern GPUs)
    "fp16": False,

    # Checkpointing & Saving
    "output_dir": "medical_qwen_cpt_v2",
    "save_strategy": "steps",
    "save_steps": 200,
    "save_total_limit": 3,  # Keep 3 most recent checkpoints

    # Evaluation
    "eval_strategy": "steps",
    "eval_steps": 100,

    # Logging
    "logging_dir": "logs_v2",
    "logging_steps": 10,

    # Data Loading (CRITICAL: avoid multiprocessing deadlock)
    "dataloader_num_workers": 0,      # NO multiprocessing (avoid freeze)
    "dataloader_pin_memory": False,   # NO memory pinning (avoid OOM)

    # Other
    "seed": 42,
}

print("\n" + "="*80)
print("TRAINING CONFIGURATION")
print("="*80)
for key, value in sorted(CONFIG.items()):
    print(f"  {key:.<50} {value}")
print("="*80 + "\n")

# Calculate effective batch size
effective_batch_size = CONFIG["per_device_train_batch_size"] * CONFIG["gradient_accumulation_steps"]
print(f"✓ Effective batch size: {effective_batch_size}")
print(f"✓ GPU memory optimizations: num_workers=0, pin_memory=False, gradient_checkpointing\n")

## 3. Load Data

In [ ]:
def load_jsonl(file_path: str, max_samples: int = None) -> List[Dict]:
    """Load JSONL file with optional sample limit."""
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if max_samples and i >= max_samples:
                break
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"Warning: Line {i} invalid JSON")
    return data

print("📂 Loading training data...")
if not Path(CONFIG["train_file"]).exists():
    print(f"✗ Error: {CONFIG['train_file']} not found!")
    print(f"Run data augmentation first: python augment_medical_data.py")
    raise FileNotFoundError(f"Training file not found: {CONFIG['train_file']}")

train_data = load_jsonl(CONFIG["train_file"])
eval_data = load_jsonl(CONFIG["eval_file"])

print(f"✓ Loaded {len(train_data):,} training chunks")
print(f"✓ Loaded {len(eval_data):,} eval chunks")

# Calculate tokens
train_tokens = sum(c.get('token_count', 0) for c in train_data)
eval_tokens = sum(c.get('token_count', 0) for c in eval_data)

print(f"\n📊 Token counts:")
print(f"  Training: {train_tokens:,} tokens")
print(f"  Eval:     {eval_tokens:,} tokens")
print(f"  Total:    {train_tokens + eval_tokens:,} tokens\n")

In [ ]:
# Show sample
print("📝 Sample training chunk:")
print("="*80)
sample = train_data[0]
print(f"Source: {sample.get('metadata', {}).get('source_book', 'N/A')}")
print(f"Augmentation: {sample.get('metadata', {}).get('augmentation', 'unknown')}")
print(f"Token count: {sample.get('token_count', 'N/A')}")
print(f"\nText (first 400 chars):")
text = sample.get('text', '')
print(text[:400] + "..." if len(text) > 400 else text)
print("="*80 + "\n")

## 4. Load Tokenizer & Model

In [ ]:
print(f"🔄 Loading tokenizer: {CONFIG['model_name']}...")
tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["model_name"],
    trust_remote_code=True,
    use_fast=True,
)

# Set pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✓ Tokenizer loaded")
print(f"  Vocab size: {len(tokenizer):,}")
print(f"  Pad token: {tokenizer.pad_token_id}")
print(f"  EOS token: {tokenizer.eos_token_id}\n")

In [ ]:
print(f"🔄 Loading model: {CONFIG['model_name']}...")
print("This may take 1-2 minutes...\n")

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True,
)

# Enable gradient checkpointing (reduce memory by ~30%)
model.gradient_checkpointing_enable()

print(f"✓ Model loaded and ready")
num_params = sum(p.numel() for p in model.parameters())
print(f"  Parameters: {num_params/1e9:.2f}B")
print(f"  Device: {model.device}")
print(f"  Dtype: {next(model.parameters()).dtype}")
print(f"  Gradient checkpointing: ✓ Enabled\n")

## 5. Tokenize Datasets

In [ ]:
def tokenize_function(examples):
    """Tokenize text with proper padding."""
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=CONFIG["max_seq_length"],
        padding="max_length",  # IMPORTANT: pad all to max_length
    )

print("🔄 Tokenizing datasets...")

# Convert to HF Dataset
train_dataset = Dataset.from_dict({"text": [c["text"] for c in train_data]})
eval_dataset = Dataset.from_dict({"text": [c["text"] for c in eval_data]})

print(f"\n  Tokenizing {len(train_dataset):,} training samples...")
train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    batch_size=100,
    remove_columns=["text"],
    desc="Train",
)

print(f"  Tokenizing {len(eval_dataset):,} eval samples...")
eval_dataset = eval_dataset.map(
    tokenize_function,
    batched=True,
    batch_size=100,
    remove_columns=["text"],
    desc="Eval",
)

print(f"\n✓ Tokenization complete")
print(f"  Train: {len(train_dataset):,} samples")
print(f"  Eval:  {len(eval_dataset):,} samples\n")

In [ ]:
# Verify tokenization
sample = train_dataset[0]
print(f"Sample token structure:")
print(f"  input_ids shape: {len(sample['input_ids'])}")
print(f"  attention_mask shape: {len(sample['attention_mask'])}")
print(f"  First 20 token IDs: {sample['input_ids'][:20]}\n")

## 6. Setup Training

In [ ]:
# Data collator for language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # Causal LM (not masked LM)
)

print("✓ Data collator configured\n")

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=CONFIG["per_device_eval_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_steps=CONFIG["warmup_steps"],
    weight_decay=CONFIG["weight_decay"],
    max_grad_norm=CONFIG["max_grad_norm"],
    bf16=CONFIG["bf16"],
    fp16=CONFIG["fp16"],
    save_strategy=CONFIG["save_strategy"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=CONFIG["save_total_limit"],
    eval_strategy=CONFIG["eval_strategy"],
    eval_steps=CONFIG["eval_steps"],
    metric_for_best_model="eval_loss",
    logging_dir=CONFIG["logging_dir"],
    logging_steps=CONFIG["logging_steps"],
    seed=CONFIG["seed"],
    dataloader_num_workers=CONFIG["dataloader_num_workers"],
    dataloader_pin_memory=CONFIG["dataloader_pin_memory"],
    load_best_model_at_end=True,
    greater_is_better=False,
    push_to_hub=False,
    report_to=["tensorboard"],
)

print("✓ Training arguments configured")
print(f"\n  Effective batch size: {effective_batch_size}")
print(f"  Total training steps: ~{(len(train_dataset) // effective_batch_size + 1) * CONFIG['num_train_epochs']:,}")
print(f"  Checkpoints will be saved every {CONFIG['save_steps']} steps")
print(f"  Evaluation every {CONFIG['eval_steps']} steps\n")

## 7. Create Trainer & Start Training

In [ ]:
# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

print("✓ Trainer created\n")
print("💾 Checkpoint directory: " + CONFIG["output_dir"])
print("📊 TensorBoard logs: " + CONFIG["logging_dir"])
print("\nTo monitor training in another terminal:")
print(f"  tensorboard --logdir {CONFIG['logging_dir']} --port 6006")
print("\n" + "="*80)
print("⚠️  TRAINING WILL NOW START - This may take 8-12 hours")
print("="*80 + "\n")

In [ ]:
# ============ START TRAINING ============
print(f"🚀 Starting training at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

try:
    train_result = trainer.train()
    print(f"\n✓ Training complete at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"\nTraining loss: {train_result.training_loss:.4f}")
except KeyboardInterrupt:
    print("\n\n⏹️  Training interrupted by user")
except Exception as e:
    print(f"\n✗ Training failed: {e}")
    import traceback
    traceback.print_exc()

## 8. Evaluate Final Model

In [ ]:
print("\n🔍 Evaluating model...\n")

eval_results = trainer.evaluate()

print("📊 Evaluation Results:")
print("="*50)
for key, value in sorted(eval_results.items()):
    if isinstance(value, float):
        print(f"  {key:.<45} {value:.4f}")
    else:
        print(f"  {key:.<45} {value}")
print("="*50 + "\n")

## 9. Save Final Model

In [ ]:
# Save best model
best_model_path = Path(CONFIG["output_dir"]) / "best_model"

print(f"💾 Saving best model to {best_model_path}...")
best_model_path.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(str(best_model_path))
tokenizer.save_pretrained(str(best_model_path))

model_size = sum(f.stat().st_size for f in best_model_path.glob('**/*')) / 1e9
print(f"✓ Model saved")
print(f"  Path: {best_model_path}")
print(f"  Size: {model_size:.2f} GB\n")

## 10. Test Inference

In [ ]:
print("🧪 Testing inference with trained model...\n")

# Load trained model
from transformers import pipeline

inference_model = AutoModelForCausalLM.from_pretrained(
    str(best_model_path),
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

generator = pipeline(
    "text-generation",
    model=inference_model,
    tokenizer=tokenizer,
    device=0,
)

# Test prompts
test_prompts = [
    "The pathophysiology of venous insufficiency involves",
    "Duplex ultrasound examination is important for diagnosing",
    "Conservative treatment options for varicose veins include",
]

print("📝 Generated Text Examples:")
print("="*80)

for i, prompt in enumerate(test_prompts, 1):
    print(f"\nExample {i}")
    print(f"Prompt: {prompt}")
    print("-" * 80)
    
    output = generator(
        prompt,
        max_length=150,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )
    
    generated = output[0]['generated_text']
    print(f"Generated: {generated}")

print("\n" + "="*80)
print("✓ Inference test complete\n")

## 11. Summary & Next Steps

In [ ]:
import glob

checkpoints = sorted(glob.glob(f"{CONFIG['output_dir']}/checkpoint-*"))

print("\n" + "="*80)
print("✓ TRAINING COMPLETE & SUCCESSFUL")
print("="*80)

print(f"\n📂 Output Files:")
print(f"  Best model:       {best_model_path}")
print(f"  Checkpoints:      {len(checkpoints)} saved")
print(f"  Latest:           {checkpoints[-1] if checkpoints else 'None'}")
print(f"  Logs:             {CONFIG['logging_dir']}/")

print(f"\n📊 Metrics:")
for key, value in sorted(eval_results.items()):
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")

print(f"\n🚀 Next Steps:")
print(f"  1. Monitor training: tensorboard --logdir {CONFIG['logging_dir']}")
print(f"  2. Load model: AutoModelForCausalLM.from_pretrained('{best_model_path}')")
print(f"  3. Push to HuggingFace Hub or deploy to production")
print(f"  4. Fine-tune further on task-specific data if needed")

print(f"\n📚 Model Info:")
print(f"  Model: {CONFIG['model_name']}")
print(f"  Base: Qwen2.5-7B (7.62B parameters)")
print(f"  CPT Data: {len(train_dataset):,} medical text samples")
print(f"  Epochs: {CONFIG['num_train_epochs']}")
print(f"  Training time: Check start time above")

print("\n" + "="*80 + "\n")